In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import cv2
import numpy as np
import hashlib
import json

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# The exact 43 source images for the V8.8 comparison
INPUT = ROOT / 'notebook55_shadow_input_43'

# Confirmed undilated authoritative masks
MASKS = ROOT / 'masks' / 'corrected'

# Compatibility outputs consumed by Notebook 09
HITL = ROOT / 'human_in_the_middle'
CONF = HITL / 'confidence_maps'
AUDIT = HITL / 'draft_audit'

CONF.mkdir(parents=True, exist_ok=True)
AUDIT.mkdir(parents=True, exist_ok=True)

SUPPORTED = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}

def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

sources = {
    p.stem: p for p in INPUT.iterdir()
    if p.is_file() and p.suffix.lower() in SUPPORTED
}

masks = {
    p.stem: p for p in MASKS.iterdir()
    if p.is_file() and p.suffix.lower() in SUPPORTED
}

missing = sorted(set(sources) - set(masks))
if missing:
    raise RuntimeError(f'Missing corrected masks: {missing}')

if len(sources) != 43:
    raise RuntimeError(
        f'Expected 43 input pages, found {len(sources)}'
    )

for page, source in sorted(sources.items()):

    mask_path = masks[page]

    image = cv2.imread(str(source), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if image is None or mask is None:
        raise RuntimeError(f'{page}: unreadable image or mask')

    if image.shape != mask.shape:
        raise RuntimeError(
            f'{page}: shape mismatch '
            f'{image.shape} vs {mask.shape}'
        )

    # Synthetic compatibility confidence only.
    # Notebook 09 requires this file for a draft audit.
    confidence = (mask > 0).astype(np.float32) * 0.20

    confidence_path = CONF / f'{page}_notebook55_compat.npy'
    np.save(confidence_path, confidence)

    record = {
        'compatibility_record': True,
        'purpose': 'Notebook55 controlled Notebook09 shadow comparison',
        'source_file': str(source),
        'source_md5': md5_file(source),
        'draft_mask': str(mask_path),
        'confidence_map': str(confidence_path),
        'confidence_is_synthetic': True,
        'mask_is_existing_authoritative_corrected_mask': True,
        'mask_is_undilated': True,
    }

    audit_path = AUDIT / f'{page}_notebook55_compat.json'

    if audit_path.exists():
        raise RuntimeError(
            f'Refusing to overwrite existing audit: {audit_path}'
        )

    audit_path.write_text(
        json.dumps(record, indent=2)
    )

print(f'Created {len(sources)} Notebook 09 compatibility records.')
print('Input:', INPUT)
print('Masks:', MASKS)
print('Audits:', AUDIT)
print('Confidence maps:', CONF)